# 🎨 Unidad 1: Color Avanzado y Aplicaciones Industriales
## Tema: Matemática del Color y Segmentación (El caso de los limones)

¡Hola! 👋 

En este taller vamos a profundizar en el **Módulo VI y VII** de la presentación. No solo vamos a ver colores, vamos a usarlos para tomar decisiones inteligentes, tal como lo haría un robot clasificador de frutas.

### Objetivos:
1.  Entender la matemática detrás de la conversión a Escala de Grises (Luminancia).
2.  Resolver el problema industrial de **"Selección de Fruta"** usando el espacio HSV para ignorar las sombras.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2 # OpenCV para la magia de visión artificial

%matplotlib inline

### 1. La Matemática del Blanco y Negro

En la diapositiva del **Módulo VI**, aparece una fórmula curiosa para convertir color a gris:

$$ Y = 0.299R + 0.587G + 0.114B $$

¿Por qué no simplemente sumamos $(R+G+B)/3$? 
Porque el ojo humano es **más sensible al verde** que al azul. Vamos a comprobarlo.

In [ ]:
# Creamos tres cuadrados de color PURO con la misma intensidad digital (255)
imagen_colores = np.zeros((100, 300, 3), dtype=np.uint8)
imagen_colores[:, :100] = [255, 0, 0]   # Rojo Puro
imagen_colores[:, 100:200] = [0, 255, 0] # Verde Puro
imagen_colores[:, 200:] = [0, 0, 255]    # Azul Puro

# MÉTODO 1: Promedio Simple (Incorrecto perceptualmente)
gris_promedio = np.mean(imagen_colores, axis=2)

# MÉTODO 2: Fórmula de Luminancia (Correcto según diapositivas)
# Y = 0.299*R + 0.587*G + 0.114*B
gris_ponderado = (
    0.299 * imagen_colores[:,:,0] + 
    0.587 * imagen_colores[:,:,1] + 
    0.114 * imagen_colores[:,:,2]
)

# Visualización
plt.figure(figsize=(10, 4))

plt.subplot(1, 3, 1)
plt.title("Original (R, G, B)")
plt.imshow(imagen_colores)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title("Promedio Simple\n(Todos se ven igual de grises)")
plt.imshow(gris_promedio, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title("Fórmula Ponderada\n(El verde se ve más brillante)")
plt.imshow(gris_ponderado, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.show()

**Conclusión:** La fórmula ponderada respeta la biología humana. El cuadrado del medio (que era verde) se ve gris claro (brillante), y el azul se ve gris oscuro. ¡Así es como vemos nosotros!

### 2. El Caso Industrial: Selección de Limones (RGB vs HSV)

En el **Módulo VII**, se menciona que HSV es ideal para segmentar frutas porque es "inmune a las sombras". Vamos a probarlo.

Voy a generar un **"Limón Sintético"**: un círculo amarillo brillante, pero con una sombra oscura en la mitad.

In [ ]:
# Función auxiliar para dibujar círculos
def crear_limon_con_sombra():
    # Fondo azulado (simulando una banda transportadora)
    img = np.zeros((200, 200, 3), dtype=np.uint8)
    img[:] = [50, 50, 100] 
    
    # Dibujamos el limón (Amarillo = Rojo + Verde)
    # Parte iluminada (Amarillo Brillante)
    cv2.circle(img, (100, 100), 60, (255, 255, 0), -1)
    
    # Parte en sombra (Amarillo Oscuro)
    # Dibujamos medio círculo oscurecido
    # En RGB, el amarillo oscuro es (100, 100, 0)
    cv2.ellipse(img, (100, 100), (60, 60), 0, 0, 180, (100, 100, 0), -1)
    
    return img

limon = crear_limon_con_sombra()

plt.figure(figsize=(4, 4))
plt.title("Limón con Sombra (Input del Robot)")
plt.imshow(limon)
plt.axis('off')
plt.show()

#### Intento 1: Segmentar en RGB
El robot busca "Amarillo". En RGB, el amarillo es (R=255, G=255). 
Pero la sombra es (R=100, G=100). 

Si le decimos al robot: "Busca píxeles donde R > 200 y G > 200", ¿qué pasará?

In [ ]:
# Umbralización en RGB
# Buscamos amarillos brillantes
mascara_rgb = cv2.inRange(limon, (150, 150, 0), (255, 255, 50))

plt.figure(figsize=(4, 4))
plt.title("Segmentación RGB (FALLO: Perdió la sombra)")
plt.imshow(mascara_rgb, cmap='gray')
plt.axis('off')
plt.show()

**Problema:** ¡El robot piensa que la mitad del limón no existe! Porque los números RGB cambiaron drásticamente con la sombra.

#### Intento 2: Segmentar en HSV
Convertimos a HSV. Aquí:
* **H (Hue):** Es el "tipo" de color. Amarillo es amarillo, sea oscuro o claro.
* **V (Value):** Es el brillo. 

Si filtramos solo por **H**, deberíamos recuperar todo el limón.

In [ ]:
# 1. Convertir a HSV
limon_hsv = cv2.cvtColor(limon, cv2.COLOR_RGB2HSV)

# 2. Ver los canales por separado
H, S, V = cv2.split(limon_hsv)

plt.figure(figsize=(10, 3))

plt.subplot(1, 3, 1)
plt.title("Canal H (Matiz)\n¡El limón es uniforme!")
plt.imshow(H, cmap='gray')

plt.subplot(1, 3, 2)
plt.title("Canal V (Brillo)\nAquí se ve la sombra")
plt.imshow(V, cmap='gray')

# 3. Segmentamos usando solo el HUE (Matiz)
# El amarillo en OpenCV suele estar alrededor de 30 (en escala 0-180)
mascara_hsv = cv2.inRange(limon_hsv, (20, 50, 50), (40, 255, 255))

plt.subplot(1, 3, 3)
plt.title("Segmentación HSV (ÉXITO)")
plt.imshow(mascara_hsv, cmap='gray')

plt.show()

### Conclusión Final del Taller

¡Lo logramos! 🎉

1.  **En RGB:** La sombra cambió los valores numéricos y el robot falló.
2.  **En HSV:** Aunque el brillo (V) cambió, el Matiz (H) se mantuvo constante. 

Por eso, como dice tu diapositiva: **"Usar HSV para segmentación"**. Es una herramienta poderosa para ingeniería porque imita cómo los humanos percibimos los objetos independientemente de la luz.